# 02_VDDC
## Variable Definition and Data Check

This notebook documents the variable definitions, missing/invalid value handling, recoding rule, and final sample size used in the analysis.

Chosen variables:
- **Behavior variable:** `SadOrHopeless`
- **Continuous variable:** `HowMuchDoYouWeighWithoutShoesInKG`


## 1. Variable Definition

### Behavior variable: `SadOrHopeless`
- **What it measures:** Whether the student felt sad or hopeless.
- **Valid codes used:** `1`, `2`
- **Meaning of success:** `success = 1`
- **Meaning of failure:** `failure = 2`
- **Recoding rule:** keep `1` as success and `2` as failure; other values are treated as invalid or missing before imputation.

### Continuous variable: `HowMuchDoYouWeighWithoutShoesInKG`
- **What it measures:** Student body weight without shoes, in kilograms.
- **Valid values used:** numeric values greater than 0
- **Invalid values:** non-numeric values or values less than or equal to 0 are treated as invalid before imputation.

### Missing or invalid values
In this notebook, missing or invalid values are handled using **Random Imputation**:
- For the behavior variable, missing/invalid values are randomly filled by sampling from the observed valid values.
- For the continuous variable, missing/invalid values are randomly filled by sampling from the observed valid values.

A fixed random seed is used so the result is reproducible.


In [49]:

import pandas as pd
import numpy as np
from pathlib import Path

input_path = Path("../data/processed/cycle2_selected_variables.csv")
output_dir1 = Path("../outputs/tables")
output_dir2 = Path("../data/processed")
output_dir1.mkdir(parents=True, exist_ok=True)
output_dir2.mkdir(parents=True, exist_ok=True)

missing_summary_path = output_dir1 / "missing_summary.csv"
output_path = output_dir2 / "cycle2_vddc_imputed.csv"


## Load data

In [50]:
df = pd.read_csv(input_path)

print("Original data shape:", df.shape)
display(df.describe().T)

Original data shape: (14041, 2)


,count,mean,std,min,25%,50%,75%,max
SadOrHopeless,13845.0,1.700036,0.458258,1.00,1.0,2.00,2.00,2.00
HowMuchDoYouWeighWithoutShoesInKG,13062.0,68.550172,16.990868,34.47,56.7,65.32,77.11,180.99


## Step 1: Identify missing / invalid values

In [51]:


behavior_col = "SadOrHopeless"
continuous_col = "HowMuchDoYouWeighWithoutShoesInKG"

# Keep original copy for checking
df_check = df.copy()

# Behavior variable: only 1 and 2 are valid
valid_behavior_codes = [1, 2]
behavior_numeric = pd.to_numeric(df_check[behavior_col], errors="coerce")
behavior_invalid_mask = ~behavior_numeric.isin(valid_behavior_codes)
behavior_missing_mask = behavior_numeric.isna()
behavior_problem_mask = behavior_invalid_mask | behavior_missing_mask
behavior_valid_mask = (~behavior_problem_mask)

# Continuous variable: numeric and > 0 are valid
continuous_numeric = pd.to_numeric(df_check[continuous_col], errors="coerce")
continuous_missing_mask = continuous_numeric.isna()
continuous_invalid_mask = continuous_numeric <= 0
continuous_problem_mask = continuous_missing_mask | continuous_invalid_mask
continuous_valid_mask = (~continuous_problem_mask)

missing_summary = pd.DataFrame({
    "variable": [behavior_col, continuous_col],
    "what_it_measures": [
        "Whether the student felt sad or hopeless",
        "Body weight without shoes in kilograms"
    ],
    "valid_codes_or_values": [
        "1, 2",
        "numeric values > 0"
    ],
    "missing_count": [
        int(behavior_missing_mask.sum()),
        int(continuous_missing_mask.sum())
    ],
    "invalid_count": [
        int((behavior_invalid_mask & ~behavior_missing_mask).sum()),
        int((continuous_invalid_mask & ~continuous_missing_mask).sum())
    ],
    "total_problem_count": [
        int(behavior_problem_mask.sum()),
        int(continuous_problem_mask.sum())
    ],
    "valid_count": [
        int(behavior_valid_mask.sum()),
        int(continuous_valid_mask.sum())
    ]
})

print("Missing / invalid value summary:")
display(missing_summary)


Missing / invalid value summary:


,variable,what_it_measures,valid_codes_or_values,missing_count,invalid_count,total_problem_count,valid_count
0,SadOrHopeless,Whether the student felt sad or hopeless,"1, 2",196,0,196,13845
1,HowMuchDoYouWeighWithoutShoesInKG,Body weight without shoes in kilograms,numeric values > 0,979,0,979,13062


## Step 2: Random imputation

In [52]:
np.random.seed(3)

df_clean = df.copy()

# Behavior random imputation
behavior_valid_values = behavior_numeric[behavior_numeric.isin(valid_behavior_codes)].dropna().astype(int).values
if len(behavior_valid_values) == 0:
    raise ValueError("No valid observed values found for SadOrHopeless. Random imputation cannot be performed.")

behavior_fill_count = int(behavior_problem_mask.sum())
if behavior_fill_count > 0:
    df_clean.loc[behavior_problem_mask, behavior_col] = np.random.choice(
        behavior_valid_values,
        size=behavior_fill_count,
        replace=True
    )

# Continuous random imputation
continuous_valid_values = continuous_numeric[(continuous_numeric > 0)].dropna().values
if len(continuous_valid_values) == 0:
    raise ValueError("No valid observed values found for HowMuchDoYouWeighWithoutShoesInKG. Random imputation cannot be performed.")

continuous_fill_count = int(continuous_problem_mask.sum())
if continuous_fill_count > 0:
    df_clean.loc[continuous_problem_mask, continuous_col] = np.random.choice(
        continuous_valid_values,
        size=continuous_fill_count,
        replace=True
    )

# Force final data types
df_clean[behavior_col] = pd.to_numeric(df_clean[behavior_col], errors="coerce").astype(int)
df_clean[continuous_col] = pd.to_numeric(df_clean[continuous_col], errors="coerce").astype(float)

print("Random imputation completed.")
print(df_clean.head())


Random imputation completed.
   SadOrHopeless  HowMuchDoYouWeighWithoutShoesInKG
0              1                              50.80
1              2                              68.04
2              1                              52.16
3              1                              79.38
4              1                              60.78


## Step 3: Final sample size used in the analysis

In [53]:
final_sample_size_behavior = int(df_clean[behavior_col].notna().sum())
final_sample_size_continuous = int(df_clean[continuous_col].notna().sum())

final_info = pd.DataFrame({
    "variable": [behavior_col, continuous_col],
    "final_sample_size_used_in_analysis": [
        final_sample_size_behavior,
        final_sample_size_continuous
    ]
})

print("Final sample size used in the analysis:")
display(final_info)


Final sample size used in the analysis:


,variable,final_sample_size_used_in_analysis
0,SadOrHopeless,14041
1,HowMuchDoYouWeighWithoutShoesInKG,14041


## Step 4: Save outputs

In [54]:
missing_summary.to_csv(missing_summary_path, index=False)
df_clean.to_csv(output_path, index=False)

print(f"Saved missing summary to: {missing_summary_path}")
print(f"Saved processed data to: {output_path}")


Saved missing summary to: ..\outputs\tables\missing_summary.csv
Saved processed data to: ..\data\processed\cycle2_vddc_imputed.csv


## 2. Short Summary

This notebook clearly shows:
- variable name
- what each variable measures
- valid codes or values used
- how missing or invalid values are handled using random imputation
- the final sample size used in the analysis

Saved outputs:
- `../data/processed/missing_summary.csv`
- `../data/processed/cycle2_vddc_imputed.csv`
